In [1]:
import pandas as pd
import numpy as np
from scipy.special import eval_legendre
from sklearn.preprocessing import MinMaxScaler

df_raw = pd.read_csv('raw_orbit_data.csv')
df_proc = df_raw.copy()

mu = 3.986004418e14  # Standart Earth gravitational parameter (m^3/s^2)
Re = 6378137.0       # Earth ekvatorial radius (m)

max_degree = 6
lifted_features = {}

lifted_features['mean_motion_n'] = np.sqrt(mu / (df_proc['a']**3))

p = df_proc['a'] * (1 - df_proc['e']**2)
J2_factor = (Re / p)**2

lifted_features['J2_raan_term'] = J2_factor * np.cos(df_proc['i'])
lifted_features['J2_omega_term'] = J2_factor * (4 - 5 * (np.sin(df_proc['i'])**2))

scaler_ae = MinMaxScaler(feature_range=(-1, 1))
df_proc[['a', 'e']] = scaler_ae.fit_transform(df_proc[['a', 'e']])

df_phys = pd.DataFrame(lifted_features)
scaler_phys = MinMaxScaler(feature_range=(-1, 1))
df_phys[df_phys.columns] = scaler_phys.fit_transform(df_phys)

math_features = {}

for col in ['a', 'e']:
    x = df_proc[col].values
    for degree in range(2, max_degree + 1):
        math_features[f"{col}_P{degree}"] = eval_legendre(degree, x)

# Fourier
for angle_col in ['i', 'omega', 'raan', 'm']:
    angle = df_proc[angle_col].values
    for k in range(1, max_degree + 1):
        math_features[f"sin_{k}{angle_col}"] = np.sin(k * angle)
        math_features[f"cos_{k}{angle_col}"] = np.cos(k * angle)

df_math = pd.DataFrame(math_features)

df_final = pd.concat([df_proc, df_phys, df_math], axis=1)

df_final.to_csv('lifted_orbit_data.csv', index=False)

print("done")

done
